# 8086 LLM - Google Colab Training Pipeline
This notebook fine-tunes the state-of-the-art **Qwen2.5-7B** model on your 8086 dataset using QLoRA and Google's free T4 GPU (15GB VRAM).

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print("Libraries loaded successfully!")

## 1. Upload Dataset
**STOP!** Before running the next cell, open the **Files** pane on the left sidebar (the folder icon) and upload your `master_dataset.txt` file.

In [ ]:
data_path = "master_dataset.txt"
if not os.path.exists(data_path):
    raise FileNotFoundError("Please upload master_dataset.txt to the sidebar first!")

with open(data_path, 'r', encoding='utf-8') as f:
    content = f.read()

examples = [chunk.strip() for chunk in content.split("<|endoftext|>") if len(chunk.strip()) > 10]
dataset = Dataset.from_dict({"text": examples})
print(f"Found {len(dataset)} examples in the dataset.")

In [ ]:
model_id = "Qwen/Qwen2.5-7B"
output_dir = "adapters"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

print(f"Loading {model_id} in 4-bit... This might take a few minutes.")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2, # T4 has 15GB VRAM, batch size 2 is safe
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=200,
    save_steps=100,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting QLoRA Fine-tuning on Google T4 GPU...")
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Training Complete!")

In [ ]:
import shutil
from google.colab import files

print("Zipping adapters for download...")
shutil.make_archive("adapters", 'zip', "adapters")
print("Downloading adapters.zip to your local computer...")
files.download("adapters.zip")